# Import

In [9]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

# Setting

In [10]:
MODEL_ID = "./base_model"     
OUT_DIR  = "./model"          

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 2048
MAX_SEQUENCE_LENGTH = 2048

# Quantization
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE = ["model.embed_tokens", "lm_head"]

# 에러가 폭발하는 레이어 지정
# Attention + MLP 전부 무시할 레이어
ignore_full_layers = list(range(26, 30))
# MLP만 무시할 레이어
ignore_mlp_layers = list()
# Attention만 무시할 레이어
ignore_attn_layers = list()

# 0 ~ 25 레이어에서 무시할 모듈
attn_modules = [
    "self_attn.q_proj",
    "self_attn.k_proj",
    "self_attn.v_proj",
    "self_attn.o_proj",
]
mlp_modules = [
    "mlp.gate_proj",
    "mlp.up_proj",
    "mlp.down_proj",
]

# 전체 보호 레이어
for layer_idx in ignore_full_layers:
    for module_name in attn_modules + mlp_modules:
        IGNORE.append(f"model.layers.{layer_idx}.{module_name}")
# MLP만 보호 레이어
for layer_idx in ignore_mlp_layers:
    for module_name in mlp_modules:
        IGNORE.append(f"model.layers.{layer_idx}.{module_name}")
# Attention만 보호 레이어
for layer_idx in ignore_attn_layers:
    for module_name in attn_modules:
        IGNORE.append(f"model.layers.{layer_idx}.{module_name}")

DAMPENING_FRAC = 0.2
BLOCK_SIZE = 128

In [11]:
import torch
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda version:", torch.version.cuda)

torch version: 2.9.1+cu130
cuda available: True
torch cuda version: 13.0


In [12]:
# GPU 메모리 상황 모니터링
from pynvml import *

nvmlInit()
handle = nvmlDeviceGetHandleByIndex(0)
info = nvmlDeviceGetMemoryInfo(handle)

print(f"Total: {info.total / 1024**2:.1f} MB")
print(f"Used : {info.used / 1024**2:.1f} MB")
print(f"Free : {info.free / 1024**2:.1f} MB")

Total: 12288.0 MB
Used : 3944.4 MB
Free : 8343.6 MB


# Model Loads

In [13]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",

    low_cpu_mem_usage=True,  # 추가
    max_memory={0: "10GiB", "cpu": "20GiB"},  # GPU 메모리 여유 확보
)

print("[INFO] 모델/토크나이저 로드 완료")

[INFO] 모델 로드 중...
[INFO] 모델/토크나이저 로드 완료


In [14]:
print("[INFO] 모델 구조 확인 중...")

# 1. 전체 구조를 트리 형태로 보기 (가장 직관적)
print(model)

print("-" * 50)

# 2. ignore에 넣을 정확한 이름(Key)만 뽑아서 보기
# (주로 Linear 레이어나 블록 단위를 확인합니다)
for name, module in model.named_modules():
    # 너무 길어지는 것을 방지하기 위해 상위 레벨만 출력하거나
    # 특정 키워드가 포함된 것만 출력할 수 있습니다.
    if "layers.0" in name or "lm_head" in name or "embed" in name:
        print(f"발견된 모듈 이름: {name}")

[INFO] 모델 구조 확인 중...
Exaone4ForCausalLM(
  (model): Exaone4Model(
    (embed_tokens): Embedding(102400, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-29): 30 x Exaone4DecoderLayer(
        (self_attn): Exaone4Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Exaone4RMSNorm((64,), eps=1e-05)
          (k_norm): Exaone4RMSNorm((64,), eps=1e-05)
        )
        (mlp): Exaone4MLP(
          (gate_proj): Linear(in_features=2048, out_features=4096, bias=False)
          (up_proj): Linear(in_features=2048, out_features=4096, bias=False)
          (down_proj): Linear(in_features=4096, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (post_attention_layernorm): Exaone4

# Dataset Loads & Preprocess

In [15]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(DATASET_ID, split=DATASET_SPLIT)
ds = ds.shuffle(seed=42).select(range(NUM_CALIBRATION_SAMPLES))

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False)
    }

ds = ds.map(preprocess)

print("[INFO] 데이터 전처리 완료")

[INFO] 캘리브레이션 데이터 로드 중...
[INFO] 데이터 전처리 완료


# GPTQ Quantization

In [16]:
print(f"[INFO] GPTQ 시작 (scheme={SCHEME}, samples={NUM_CALIBRATION_SAMPLES}, max_len={MAX_SEQUENCE_LENGTH})...")

# 양자화 전 메모리 정리
import gc
torch.cuda.empty_cache()
gc.collect()

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
        dampening_frac=DAMPENING_FRAC,
        block_size=BLOCK_SIZE,
    )
]

# GPTQ 시작 전에 추가
def print_gpu_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        reserved = torch.cuda.memory_reserved(0) / 1024**3
        print(f"[MEM] Allocated: {allocated:.2f}GB, Reserved: {reserved:.2f}GB")

print_gpu_memory()

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,

    batch_size=1,  # 배치 크기 최소화
    
    # 데이터 처리 최적화
    text_column="text",
    pad_to_max_length=False,  # 패딩 비활성화로 메모리 절약
    shuffle_calibration_samples=True,
    concatenate_data=False,
    
    # 캐시 및 전처리
    overwrite_cache=True,
    preprocessing_num_workers=1,  # 워커 수 제한
    
    # 양자화 설정
    quantization_aware_calibration=True,
)

print_gpu_memory()

print("[INFO] GPTQ 완료")

[INFO] GPTQ 시작 (scheme=W4A16, samples=2048, max_len=2048)...
[MEM] Allocated: 4.77GB, Reserved: 4.77GB


Tokenizing (num_proc=1): 100%|██████████| 2048/2048 [00:03<00:00, 677.74 examples/s]

2026-02-12T14:23:01.608779+0900 | reset | INFO - Compression lifecycle reset
2026-02-12T14:23:01.610392+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-12T14:23:01.649852+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-12T14:23:01.650386+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`



(1/31): Calibrating: 100%|██████████| 2048/2048 [00:13<00:00, 146.89it/s]

2026-02-12T14:23:17.786747+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 2048 samples


...

(26/31): Calibrating: 100%|██████████| 2048/2048 [00:15<00:00, 132.91it/s]

2026-02-12T14:35:00.452489+0900 | compress_modules | INFO - Quantizing model.layers.25.self_attn.q_proj using 2048 samples


2026-02-12T14:35:00.828826+0900 | compress | METRIC - time 0.38s
2026-02-12T14:35:00.829666+0900 | compress | METRIC - error 3996.14
2026-02-12T14:35:00.830242+0900 | compress | METRIC - GPU 0 | usage: 38.15% | total memory: 12 GB
2026-02-12T14:35:00.830604+0900 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-12T14:35:00.831282+0900 | compress_modules | INFO - Quantizing model.layers.25.self_attn.k_proj using 2048 samples
2026-02-12T14:35:01.188050+0900 | compress | METRIC - time 0.36s
2026-02-12T14:35:01.188913+0900 | compress | METRIC - error 1023.95
2026-02-12T14:35:01.189250+0900 | compress | METRIC - GPU 0 | usage: 38.15% | total memory: 12 GB
2026-02-12T14:35:01.189562+0900 | compress | METRIC - Compressed module size: 2.121728 MB
2026-02-12T14:35:01.189885+0900 | compress_modules | INFO - Quantizing model.layers.25.self_attn.v_proj using 2048 samples
2026-02-12T14:35:01.543400+0900 | compress | METRIC - time 0.35s
2026-02-12T14:35:01.544389+0900 | compress | ME

(31/31): Propagating: 100%|██████████| 2048/2048 [00:03<00:00, 679.69it/s]

2026-02-12T14:36:22.724709+0900 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-12T14:36:22.748037+0900 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
[MEM] Allocated: 2.39GB, Reserved: 2.80GB
[INFO] GPTQ 완료


# Test

In [17]:
# ==========================================
# [검증 코드] 양자화된 모델 성능 & 속도 테스트
# ==========================================
import time
import torch
from torch.nn import CrossEntropyLoss
from tqdm import tqdm

print("\n[INFO] 검증 시작...")

# 1. 모델을 평가 모드로 전환
model.eval()

# ------------------------------------------------------------------
# 테스트 1: 정성 평가 (실제 대화 생성) - 모델이 깨졌는지 눈으로 확인
# ------------------------------------------------------------------
print("\n=== [1] 생성 테스트 (Qualitative Test) ===")
test_prompts = [
    "인공지능의 미래에 대해 설명해줘.",
    "1+1은 뭐야?", 
    "대한민국의 수도는 어디야?"
]

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # 시간 측정 시작
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=50,      # 짧게 생성
            do_sample=False,        # 결정론적 생성 (Greedy)
            pad_token_id=tokenizer.eos_token_id
        )
    end_time = time.time()
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    tokens_generated = len(outputs[0]) - inputs['input_ids'].shape[1]
    tps = tokens_generated / (end_time - start_time)
    
    print(f"Q: {prompt}")
    print(f"A: {generated_text}")
    print(f"-> 속도: {tps:.2f} tokens/sec\n")

# ------------------------------------------------------------------
# 테스트 2: 정량 평가 (Perplexity - PPL) - 점수(Score) 예측 지표
# PPL이 낮을수록 좋음. (Base Model 대비 너무 높으면 망한 것)
# ------------------------------------------------------------------
print("=== [2] PPL(Perplexity) 테스트 (Quantitative Test) ===")

def calculate_ppl(model, tokenizer, text_list, max_length=2048):
    # 메모리 정리를 위해 grad 비활성화
    model.eval()
    nlls = []
    total_tokens = 0
    
    loss_fct = CrossEntropyLoss()

    print(f"-> {len(text_list)}개의 샘플로 PPL 계산 중...")
    
    with torch.no_grad():
        for text in tqdm(text_list):
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(model.device)
            
            # 라벨은 input_ids와 동일하게 설정 (Self-Supervised Learning)
            output = model(input_ids=inputs.input_ids, labels=inputs.input_ids)
            loss = output.loss
            
            # Loss 누적
            nlls.append(loss.item() * inputs.input_ids.shape[1])
            total_tokens += inputs.input_ids.shape[1]

    # 평균 Loss 계산
    avg_loss = sum(nlls) / total_tokens
    ppl = torch.exp(torch.tensor(avg_loss))
    return ppl.item()

# 검증용 데이터 소량 추출 (학습에 안 쓴 데이터면 더 좋지만, 여기선 빠른 확인을 위해 train 앞부분 사용)
# *중요*: oneshot에 쓴 데이터와 안 겹치는 부분을 쓰는게 정확하지만, 대략적인 파괴 여부 확인용임
val_ds = load_dataset(DATASET_ID, split="train").select(range(NUM_CALIBRATION_SAMPLES, NUM_CALIBRATION_SAMPLES + 30))
val_texts = [
    tokenizer.apply_chat_template(x["conversations"], tokenize=False, add_generation_prompt=True) 
    for x in val_ds
]

try:
    ppl_score = calculate_ppl(model, tokenizer, val_texts)
    print(f"\n★ 예측 Perplexity (PPL): {ppl_score:.4f}")
    
    if ppl_score < 10:
        print("-> [상태: 좋음] 모델이 잘 보존되었습니다. (리더보드 점수 기대 가능)")
    elif ppl_score < 20:
        print("-> [상태: 주의] 성능 저하가 조금 있습니다. (파라미터 튜닝 필요)")
    else:
        print("-> [상태: 위험] 모델이 많이 손상되었습니다. (dampening_frac 높이거나 group_size 확인)")

except Exception as e:
    print(f"PPL 계산 중 오류 발생: {e}")

# 메모리 정리
torch.cuda.empty_cache()


[INFO] 검증 시작...

=== [1] 생성 테스트 (Qualitative Test) ===
Q: 인공지능의 미래에 대해 설명해줘.
A: 인공지능의 미래에 대해 설명해줘.
-> 속도: 0.59 tokens/sec

Q: 1+1은 뭐야?
A: 1+1은 뭐야?
-> 속도: 0.60 tokens/sec

Q: 대한민국의 수도는 어디야?
A: 대한민국의 수도는 어디야?
</think>
-> 속도: 0.63 tokens/sec

=== [2] PPL(Perplexity) 테스트 (Quantitative Test) ===
-> 30개의 샘플로 PPL 계산 중...


100%|██████████| 30/30 [11:01<00:00, 22.06s/it]


★ 예측 Perplexity (PPL): 4.6794
-> [상태: 좋음] 모델이 잘 보존되었습니다. (리더보드 점수 기대 가능)


In [18]:
# ==========================================
# 성능 평가 및 점수 계산 (데이터셋 재사용 버전)
# ==========================================
import math

# 함수 인자 변경: dataset_split -> dataset
def evaluate_model_performance(model, tokenizer, dataset, num_samples=30):
    """
    미리 로드된 dataset의 뒷부분 데이터를 사용하여 PPL과 Latency를 측정합니다.
    """
    model.eval()
    
    # 1. 검증 데이터 준비 (이미 만들어진 ds의 뒷부분 num_samples개 사용)
    # 예: 총 1024개면, 994번 ~ 1023번 데이터를 사용
    total_len = len(dataset)
    start_idx = max(0, total_len - num_samples)
    
    # 데이터셋 슬라이싱 (select 사용)
    val_ds = dataset.select(range(start_idx, total_len))
    
    # 이미 전처리(preprocess)가 되어 있으므로 "text" 컬럼을 그대로 사용
    val_texts = val_ds["text"]

    # 2. PPL 측정
    nlls = []
    total_tokens_ppl = 0
    
    print(f"\n[Eval] PPL 측정 중... (Dataset Index: {start_idx}~{total_len-1}, {len(val_texts)}개)")
    
    with torch.no_grad():
        for text in tqdm(val_texts, desc="PPL"):
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
            output = model(input_ids=inputs.input_ids, labels=inputs.input_ids)
            nlls.append(output.loss.item() * inputs.input_ids.shape[1])
            total_tokens_ppl += inputs.input_ids.shape[1]
    
    avg_loss = sum(nlls) / total_tokens_ppl
    ppl = math.exp(avg_loss)

    # 3. 속도 측정 (기존과 동일)
    test_prompt = "인공지능의 미래에 대해 설명해줘."
    inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
    
    print(f"[Eval] 추론 속도(Latency) 측정 중...")
    
    # 워밍업
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    
    # 실제 측정
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=100, 
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    end_time = time.time()
    
    generated_tokens = len(outputs[0]) - inputs['input_ids'].shape[1]
    total_time = end_time - start_time
    seconds_per_token = total_time / generated_tokens
    
    return ppl, seconds_per_token

# ==========================================
# 실행 부분 (수정됨)
# ==========================================

print("\n[INFO] Quantized Model 평가 시작...")

# 평가 수행
quant_ppl, quant_latency = evaluate_model_performance(model, tokenizer, dataset=ds, num_samples=30)

# 기준값 설정 (목표치)
TARGET_PPL = 5.5       # 기준 모델 PPL
TARGET_LATENCY = 2.0   # 기준 모델 속도

ppl_score = 0.5 * quant_ppl / TARGET_PPL
speed_score = 0.5 * quant_latency / TARGET_LATENCY

total_score = ppl_score + speed_score

print("\n" + "="*50)
print("             🏆 리더보드 결과             ")
print("="*50)
print(f"1. Model Stats")
print(f"   - PPL       : {quant_ppl:.4f}")
print(f"   - Latency   : {quant_latency:.4f} sec/token")
print("-" * 50)
print(f"2. Score Components (Weight 0.5 each)")
print(f"   - PPL Score  : {ppl_score:.4f}")
print(f"   - Speed Score : {speed_score:.4f}")
print("-" * 50)
print(f"★ Total Score (PPL Score + Speed Score) : {total_score:.4f}")
print("="*50)


[INFO] Quantized Model 평가 시작...

[Eval] PPL 측정 중... (Dataset Index: 2018~2047, 30개)


PPL: 100%|██████████| 30/30 [09:06<00:00, 18.20s/it]


[Eval] 추론 속도(Latency) 측정 중...

             🏆 리더보드 결과             
1. Model Stats
   - PPL       : 4.2754
   - Latency   : 1.7207 sec/token
--------------------------------------------------
2. Score Components (Weight 0.5 each)
   - PPL Score  : 0.3887
   - Speed Score : 0.4302
--------------------------------------------------
★ Total Score (PPL Score + Speed Score) : 0.8189


# Model Save

In [19]:
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"[INFO] 모델 저장 완료: {OUT_DIR}")

2026-02-12T14:56:45.819751+0900 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 182it [00:02, 82.58it/s]


[INFO] 모델 저장 완료: ./model


# Submission

In [20]:
zip_name = "submit-ver24"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

print(f"[INFO] 생성 완료: {zip_name}.zip")

[INFO] submit-ver24.zip 생성 중...
[INFO] 생성 완료: submit-ver24.zip
